# Day 40 — Offline Integrated Pipeline v0.1 (Standalone Walkthrough v2)

Notebook này là **reference walkthrough có thể chạy độc lập**, không thay thế CLI/service implementation chính.

Pipeline được kiểm thử:

```text
input manifest
→ governance preflight
→ ingestion
→ protocol mapping
→ L0–L5 quality gate
→ preprocessing
→ windowing
→ Feature Set 14
→ governed inference
→ calibration / abstention
→ Task C supportability
→ fatigue-context engine
→ report + provenance
```

**Safety boundary**
- inference only; không fit model/scaler/calibrator/threshold;
- unsupported protocol là hard eligibility gate;
- giữ `raw_model_confidence` và `calibrated_confidence` cho audit;
- đặt `effective_confidence=0.0`, `interpretation_allowed=false` khi unsupported protocol;
- quality fail không đồng nghĩa “không phát hiện mỏi”;
- không hard fatigue diagnosis hoặc treatment recommendation.

Mặc định notebook chạy synthetic fixtures. `NORMALIZED_REAL` chỉ chạy khi có governed bundle Day 39 và normalized input phù hợp.

## Cell 1 — Environment and governance

**Input:** Python/Colab runtime.  
**Output:** deterministic pipeline configuration.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import joblib
import numpy as np
import pandas as pd
from scipy import signal

RUN_MODE = os.getenv('DAY40_RUN_MODE', 'SYNTHETIC').upper()
assert RUN_MODE in {'SYNTHETIC', 'NORMALIZED_REAL'}

MODEL_FITTING_ALLOWED = False
SCALER_FITTING_ALLOWED = False
CALIBRATOR_FITTING_ALLOWED = False
THRESHOLD_FITTING_ALLOWED = False
SEALED_TEST_OPENED = False
HARD_FATIGUE_DIAGNOSIS_ALLOWED = False
TREATMENT_RECOMMENDATION_ALLOWED = False

assert not MODEL_FITTING_ALLOWED
assert not SCALER_FITTING_ALLOWED
assert not CALIBRATOR_FITTING_ALLOWED
assert not THRESHOLD_FITTING_ALLOWED
assert not SEALED_TEST_OPENED
assert not HARD_FATIGUE_DIAGNOSIS_ALLOWED
assert not TREATMENT_RECOMMENDATION_ALLOWED

SEED = 4001
rng = np.random.default_rng(SEED)
print({'python': platform.python_version(), 'mode': RUN_MODE, 'seed': SEED})

{'python': '3.12.13', 'mode': 'SYNTHETIC', 'seed': 4001}


## Cell 2 — Paths, governed bundle and output contract

**Input:** Day 39 governance manifest, model bundle and Day 35 calibration bundle.  
**Output:** frozen path contract.

In [2]:
DRIVE_ROOT = Path('/content/drive/MyDrive/MyoLab-AI-data')
OUTPUT_ROOT = Path('/content/day40-offline-pipeline-v2') if RUN_MODE == 'SYNTHETIC' else DRIVE_ROOT / 'offline-pipeline/day40-v0.1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DAY39_MANIFEST = DRIVE_ROOT / 'governance/day39/day39-governance-readiness.json'
MODEL_BUNDLE_PATH = DRIVE_ROOT / 'governance/day39/governed-model-bundle.joblib'
DAY35_BUNDLE_PATH = DRIVE_ROOT / 'governance/day39/day35-calibration-abstention-bundle.joblib'

PIPELINE_CONFIG = {
    'pipeline_version': 'offline-pipeline-v0.1',
    'protocols': {
        'synthetic-gesture-primary4.v1': {
            'supported': True,
            'sampling_rate_hz': 2000,
            'active_phase_seconds': [0.0, 5.0],
            'window_ms': 200,
            'hop_ms': 100,
            'channel_count': 3,
        },
        'unsupported-demo.v1': {'supported': False},
    },
    'qc': {
        'sampling_rate_tolerance_fraction': 0.01,
        'nonfinite_fail_ratio': 0.01,
        'flatline_std_threshold': 1e-10,
        'privacy_fields': ['patient_name', 'mrn', 'date_of_birth'],
    },
    'preprocessing': {
        'policy_id': 'day40-bandpass20-450-notch50-dc-v1',
        'bandpass_hz': [20.0, 450.0],
        'notch_hz': 50.0,
        'notch_q': 30.0,
    },
    'features': {
        'feature_set_version': 'feature-set-14.v1.0.0',
        'feature_order': ['MAV','RMS','WL','ZC','SSC','WAMP','VAR','IEMG','MNF','MDF','PKF','SM1','SM2','SM3'],
    },
    'context': {
        'unsupported_protocol_hard_gate': True,
        'quality_fail_hard_gate': True,
        'confidence_may_increase': False,
        'hard_fatigue_diagnosis_allowed': False,
    },
    'report': {
        'not_approved_for_patient_use': True,
        'decision_support_only': True,
        'no_automated_treatment_recommendation': True,
    },
}

print(json.dumps({'output_root': str(OUTPUT_ROOT), 'run_mode': RUN_MODE}, indent=2))

{
  "output_root": "/content/day40-offline-pipeline-v2",
  "run_mode": "SYNTHETIC"
}


## Cell 3 — Schemas, state machine and provenance helpers

**Input:** stage payloads.  
**Output:** deterministic state transitions and artifact ledger.

In [3]:
STATES = [
    'RECEIVED', 'PREFLIGHT_PASSED', 'IMPORTED', 'QC_PASSED', 'QC_WARNING',
    'QC_FAILED', 'FEATURES_READY', 'INFERENCE_READY', 'ABSTAINED',
    'ANALYZED', 'REPORTED', 'FAILED',
]

ALLOWED_TRANSITIONS = {
    'RECEIVED': {'PREFLIGHT_PASSED', 'FAILED'},
    'PREFLIGHT_PASSED': {'IMPORTED', 'FAILED'},
    'IMPORTED': {'QC_PASSED', 'QC_WARNING', 'QC_FAILED', 'FAILED'},
    'QC_PASSED': {'FEATURES_READY', 'FAILED'},
    'QC_WARNING': {'FEATURES_READY', 'FAILED'},
    'QC_FAILED': {'ABSTAINED', 'FAILED'},
    'FEATURES_READY': {'INFERENCE_READY', 'FAILED'},
    'INFERENCE_READY': {'ABSTAINED', 'ANALYZED', 'FAILED'},
    'ABSTAINED': {'ANALYZED', 'REPORTED', 'FAILED'},
    'ANALYZED': {'REPORTED', 'FAILED'},
    'REPORTED': set(),
    'FAILED': set(),
}

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(payload: bytes):
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def json_safe(v):
    if isinstance(v, Path): return str(v)
    if isinstance(v, np.generic): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, dict): return {str(k): json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [json_safe(x) for x in v]
    return v

def write_json(path: Path, payload: dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding='utf-8')

def transition(run, new_state, reason=None):
    current = run['state']
    if new_state not in ALLOWED_TRANSITIONS[current]:
        raise RuntimeError(f'Illegal transition {current} -> {new_state}')
    run['transitions'].append({
        'from': current,
        'to': new_state,
        'timestamp_utc': utc_now_iso(),
        'reason': reason,
    })
    run['state'] = new_state

@dataclass
class StageResult:
    stage: str
    status: str
    reason_codes: list[str] = field(default_factory=list)
    payload: dict[str, Any] = field(default_factory=dict)
    duration_ms: float = 0.0

print('[PASS] State machine loaded.')

[PASS] State machine loaded.


## Cell 4 — Canonical Feature Set 14

**Input:** preprocessed multi-channel sEMG windows.  
**Output:** exact per-channel TD8 + SP6 features.

In [4]:
FEATURE_ORDER = PIPELINE_CONFIG['features']['feature_order']

def safe_crossings(x, threshold=0.0):
    centered = x - threshold
    return int(np.sum(centered[:-1] * centered[1:] < 0))

def spectral_features(x, fs):
    freqs, psd = signal.welch(x, fs=fs, nperseg=min(len(x), 256), detrend=False)
    psd = np.maximum(psd, 0)
    power = float(np.sum(psd))
    if power <= 1e-18:
        return [np.nan] * 6
    mnf = float(np.sum(freqs * psd) / power)
    csum = np.cumsum(psd)
    mdf = float(freqs[np.searchsorted(csum, csum[-1] / 2.0)])
    pkf = float(freqs[np.argmax(psd)])
    sm1 = float(np.sum(freqs * psd))
    sm2 = float(np.sum((freqs ** 2) * psd))
    sm3 = float(np.sum((freqs ** 3) * psd))
    return [mnf, mdf, pkf, sm1, sm2, sm3]

def feature14(x, fs):
    x = np.asarray(x, dtype=float)
    diff = np.diff(x)
    second = np.diff(x, n=2)
    threshold = 1e-4 * max(np.std(x), 1e-12)
    td = [
        float(np.mean(np.abs(x))),
        float(np.sqrt(np.mean(x*x))),
        float(np.sum(np.abs(diff))),
        safe_crossings(x, 0.0),
        int(np.sum((second[:-1] * second[1:] < 0) & (np.abs(second[:-1] - second[1:]) > threshold))) if len(second) > 1 else 0,
        int(np.sum(np.abs(diff) > threshold)),
        float(np.var(x, ddof=1)) if len(x) > 1 else 0.0,
        float(np.sum(np.abs(x))),
    ]
    return np.asarray(td + spectral_features(x, fs), dtype=float)

def preprocess_signal(x, fs):
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x, axis=0, keepdims=True)
    low, high = PIPELINE_CONFIG['preprocessing']['bandpass_hz']
    if high >= fs / 2:
        high = 0.95 * fs / 2
    sos = signal.butter(4, [low, high], btype='bandpass', fs=fs, output='sos')
    y = signal.sosfiltfilt(sos, x, axis=0)
    notch_hz = PIPELINE_CONFIG['preprocessing']['notch_hz']
    if notch_hz < fs / 2:
        b, a = signal.iirnotch(notch_hz, PIPELINE_CONFIG['preprocessing']['notch_q'], fs)
        y = signal.filtfilt(b, a, y, axis=0)
    return y

def window_and_extract(x, fs, window_ms, hop_ms):
    window_samples = int(round(window_ms * fs / 1000.0))
    hop_samples = int(round(hop_ms * fs / 1000.0))
    rows = []
    for start in range(0, len(x) - window_samples + 1, hop_samples):
        stop = start + window_samples
        feats = np.concatenate([feature14(x[start:stop, ch], fs) for ch in range(x.shape[1])])
        rows.append({'start': start, 'stop': stop, 'features': feats})
    return rows

# Unit checks
z = np.zeros(400)
assert np.isnan(feature14(z, 2000)[8:]).all()
t = np.arange(0, 0.2, 1/2000)
s = np.sin(2*np.pi*100*t)
f = feature14(s, 2000)
assert 80 <= f[8] <= 120
print('[PASS] Feature Set 14 unit checks.')

[PASS] Feature Set 14 unit checks.


## Cell 5 — Governed model and calibration bundles

**Input:** Day 39 bundle in real mode, fixed synthetic bundle otherwise.  
**Output:** inference-only bundle with hashes and thresholds.

In [5]:
def make_synthetic_bundle():
    class_order = ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
    n_features = 3 * 14
    weights = rng.normal(0, 0.12, (len(class_order), n_features))
    # Hand-authored deterministic separation for smoke scenarios; no fitting occurs.
    for i in range(len(class_order)):
        weights[i, i::4] += 0.35
    return {
        'bundle_version': 'synthetic-governed-bundle-v1',
        'registry_state': 'research',
        'rerun_status': 'EXACT_MATCH',
        'model_card_present': True,
        'class_order': class_order,
        'expected_feature_dim': n_features,
        'scaler_mean': np.zeros(n_features),
        'scaler_scale': np.ones(n_features),
        'linear_weights': weights,
        'linear_bias': np.zeros(len(class_order)),
        'temperature': 1.25,
        'confidence_score': 'max_probability',
        'abstention_threshold': 0.45,
        'feature_set_version': PIPELINE_CONFIG['features']['feature_set_version'],
    }

if RUN_MODE == 'SYNTHETIC':
    governed_bundle = make_synthetic_bundle()
    day39_gate = {
        'status': 'SYNTHETIC_GOVERNED_FIXTURE',
        'registry_state': 'research',
        'rerun_status': 'EXACT_MATCH',
        'model_card_present': True,
    }
else:
    for path in [DAY39_MANIFEST, MODEL_BUNDLE_PATH, DAY35_BUNDLE_PATH]:
        if not path.exists():
            raise FileNotFoundError(path)
    day39_gate = json.loads(DAY39_MANIFEST.read_text(encoding='utf-8'))
    if day39_gate.get('status') not in {'GO_FOR_DAY40_OFFLINE_PIPELINE_V0_1', 'GO_FOR_DAY40_WITH_GOVERNANCE_WARNINGS'}:
        raise RuntimeError(f'Day39 gate not eligible: {day39_gate.get("status")}')
    governed_bundle = joblib.load(MODEL_BUNDLE_PATH)
    calibration_bundle = joblib.load(DAY35_BUNDLE_PATH)
    governed_bundle['temperature'] = calibration_bundle['temperature']
    governed_bundle['abstention_threshold'] = calibration_bundle['threshold']

required_bundle_fields = {
    'registry_state', 'rerun_status', 'model_card_present', 'class_order',
    'expected_feature_dim', 'temperature', 'abstention_threshold',
    'feature_set_version',
}
missing = required_bundle_fields - set(governed_bundle)
if missing:
    raise RuntimeError(f'Model bundle missing fields: {sorted(missing)}')
if governed_bundle['registry_state'] != 'research':
    raise RuntimeError('Only research-state bundle is allowed in Day40 technical pipeline.')
if governed_bundle['rerun_status'] not in {'EXACT_MATCH', 'MATCH_WITHIN_TOLERANCE'}:
    raise RuntimeError('Governed bundle rerun gate failed.')
print('[PASS] Governed bundle loaded.')

[PASS] Governed bundle loaded.


## Cell 6 — Offline pipeline orchestrator

**Input:** canonical input manifest and signal payload.  
**Output:** deterministic stage results, context decision and report artifacts.

In [6]:
def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def infer_features(X):
    if 'linear_weights' in governed_bundle:
        mean = np.asarray(governed_bundle['scaler_mean'])
        scale = np.asarray(governed_bundle['scaler_scale'])
        Z = (X - mean) / np.where(scale == 0, 1.0, scale)
        logits = Z @ np.asarray(governed_bundle['linear_weights']).T + np.asarray(governed_bundle['linear_bias'])
        raw_prob = softmax(logits)
    else:
        model = governed_bundle['model']
        scaler = governed_bundle['scaler']
        Z = scaler.transform(X)
        if hasattr(model, 'predict_proba'):
            raw_prob = model.predict_proba(Z)
        else:
            scores = model.decision_function(Z)
            if scores.ndim == 1:
                scores = np.column_stack([-scores, scores])
            raw_prob = softmax(scores)
    temperature = float(governed_bundle['temperature'])
    calibrated = softmax(np.log(np.clip(raw_prob, 1e-12, 1.0)) / temperature)
    return raw_prob, calibrated

class OfflinePipeline:
    def __init__(self, config, bundle, output_root):
        self.config = config
        self.bundle = bundle
        self.output_root = Path(output_root)

    def run(self, manifest, signal_payload):
        run = {
            'run_id': f'D40-{uuid.uuid4().hex[:12]}',
            'state': 'RECEIVED',
            'transitions': [],
            'started_at_utc': utc_now_iso(),
            'stages': [],
        }
        session_dir = self.output_root / manifest['session_id']
        for sub in ['manifests','qc','features','inference','taskc','context','report','logs']:
            (session_dir / sub).mkdir(parents=True, exist_ok=True)
        try:
            # S0 preflight
            required_manifest = {
                'session_id','subject_id','dataset_id','source_type','sampling_rate_hz',
                'channel_metadata','protocol_id','protocol_version','task_label',
                'active_phase','unit','privacy_scan_status',
            }
            missing = required_manifest - set(manifest)
            if missing:
                raise RuntimeError(f'MISSING_MANIFEST_FIELDS:{sorted(missing)}')
            if manifest['privacy_scan_status'] != 'pass':
                raise RuntimeError('PRIVACY_SCAN_NOT_PASS')
            transition(run, 'PREFLIGHT_PASSED')
            write_json(session_dir / 'manifests/input-manifest.json', manifest)

            # S1 ingestion
            x = np.asarray(signal_payload, dtype=float)
            if x.ndim != 2 or x.size == 0:
                raise RuntimeError('IMPORT_REJECTED_INVALID_SIGNAL_SHAPE')
            transition(run, 'IMPORTED')

            # S2 protocol mapping + S3 QC
            protocol_key = f"{manifest['protocol_id']}.{manifest['protocol_version']}"
            protocol = self.config['protocols'].get(protocol_key)
            protocol_supported = bool(protocol and protocol.get('supported', False))
            nonfinite_ratio = float(np.mean(~np.isfinite(x)))
            flat_channels = [int(i) for i in range(x.shape[1]) if np.nanstd(x[:, i]) <= self.config['qc']['flatline_std_threshold']]
            qc_reasons = []
            if nonfinite_ratio > self.config['qc']['nonfinite_fail_ratio']:
                qc_reasons.append('NONFINITE_RATIO_FAIL')
            if flat_channels:
                qc_reasons.append('FLATLINE_CHANNEL')
            expected_fs = protocol.get('sampling_rate_hz') if protocol else None
            if expected_fs is not None:
                frac = abs(manifest['sampling_rate_hz'] - expected_fs) / expected_fs
                if frac > self.config['qc']['sampling_rate_tolerance_fraction']:
                    qc_reasons.append('SAMPLING_RATE_MISMATCH')
            quality_status = 'fail' if qc_reasons else ('warning' if manifest.get('force_warning', False) else 'pass')
            qc_report = {
                'quality_status': quality_status,
                'reason_codes': qc_reasons,
                'nonfinite_ratio': nonfinite_ratio,
                'flat_channels': flat_channels,
                'protocol_supported': protocol_supported,
            }
            write_json(session_dir / 'qc/quality-report.json', qc_report)
            if quality_status == 'fail':
                transition(run, 'QC_FAILED', ';'.join(qc_reasons))
                transition(run, 'ABSTAINED', 'ABSTAIN_QUALITY_FAIL')
                raw_conf = calibrated_conf = effective_conf = 0.0
                prediction = None
                decision_status = 'ABSTAIN_QUALITY_FAIL'
                interpretation_allowed = False
                feature_summary = {'status': 'BLOCKED_BY_QUALITY_GATE'}
            else:
                transition(run, 'QC_WARNING' if quality_status == 'warning' else 'QC_PASSED')
                # S4-S7 only for quality pass/warning and supported extraction contract
                if protocol is None:
                    protocol = {'window_ms': 200, 'hop_ms': 100, 'sampling_rate_hz': manifest['sampling_rate_hz']}
                fs = float(manifest['sampling_rate_hz'])
                y = preprocess_signal(np.nan_to_num(x), fs)
                windows = window_and_extract(y, fs, protocol.get('window_ms', 200), protocol.get('hop_ms', 100))
                Xf = np.asarray([w['features'] for w in windows])
                if Xf.shape[1] != int(self.bundle['expected_feature_dim']):
                    raise RuntimeError(f'FEATURE_DIMENSION_MISMATCH:{Xf.shape[1]}')
                if not np.isfinite(Xf).all():
                    raise RuntimeError('NONFINITE_FEATURE_MATRIX')
                transition(run, 'FEATURES_READY')
                feature_summary = {
                    'status': 'READY',
                    'window_count': len(Xf),
                    'feature_dimension': Xf.shape[1],
                    'aggregation': 'median_probability_across_windows',
                }
                write_json(session_dir / 'features/feature-summary.json', feature_summary)

                raw_prob, calibrated_prob = infer_features(Xf)
                mean_raw = np.mean(raw_prob, axis=0)
                mean_cal = np.mean(calibrated_prob, axis=0)
                class_order = list(self.bundle['class_order'])
                pred_idx = int(np.argmax(mean_cal))
                prediction = class_order[pred_idx]
                raw_conf = float(np.max(mean_raw))
                calibrated_conf = float(np.max(mean_cal))
                transition(run, 'INFERENCE_READY')

                # S8/S10: hard eligibility gates precede confidence policy.
                if not protocol_supported:
                    decision_status = 'ABSTAIN_UNSUPPORTED_PROTOCOL'
                    effective_conf = 0.0
                    interpretation_allowed = False
                    transition(run, 'ABSTAINED', decision_status)
                elif calibrated_conf < float(self.bundle['abstention_threshold']):
                    decision_status = 'ABSTAIN_LOW_CONFIDENCE'
                    effective_conf = 0.0
                    interpretation_allowed = False
                    transition(run, 'ABSTAINED', decision_status)
                else:
                    decision_status = 'ACCEPTED_RESEARCH_PREDICTION'
                    effective_conf = calibrated_conf
                    interpretation_allowed = True
                    transition(run, 'ANALYZED')

            # S9 Task C supportability
            taskc = {
                'repeatability': 'INSUFFICIENT_REPETITIONS',
                'similarity': 'INSUFFICIENT_REPETITIONS',
                'cocontraction': 'NOT_ELIGIBLE_ANATOMICAL_MAPPING_UNVERIFIED',
                'numeric_zero_used_for_ineligible': False,
            }
            write_json(session_dir / 'taskc/quantitative-summary.json', taskc)

            # Context result: no hard diagnosis.
            context_state = (
                'QUALITY_BLOCKED' if decision_status == 'ABSTAIN_QUALITY_FAIL'
                else 'UNSUPPORTED_PROTOCOL' if decision_status == 'ABSTAIN_UNSUPPORTED_PROTOCOL'
                else 'LOW_CONFIDENCE' if decision_status == 'ABSTAIN_LOW_CONFIDENCE'
                else 'NO_FATIGUE_CONTEXT_CLAIM'
            )
            context = {
                'context_state': context_state,
                'raw_model_confidence': raw_conf,
                'calibrated_confidence': calibrated_conf,
                'effective_confidence': effective_conf,
                'decision_status': decision_status,
                'interpretation_allowed': interpretation_allowed,
                'hard_fatigue_diagnosis': None,
                'reason_codes': [decision_status] if decision_status.startswith('ABSTAIN') else [],
                'confidence_may_increase': False,
            }
            write_json(session_dir / 'context/context-result.json', context)

            inference = {
                'prediction': prediction if interpretation_allowed else None,
                'audit_prediction_before_gate': prediction,
                **context,
            }
            write_json(session_dir / 'inference/prediction.json', inference)

            if run['state'] in {'ABSTAINED', 'ANALYZED'}:
                transition(run, 'REPORTED')
            report = {
                'session_id': manifest['session_id'],
                'protocol': protocol_key,
                'quality_outcome': quality_status,
                'model_status': decision_status,
                'predicted_class': prediction if interpretation_allowed else None,
                'calibrated_confidence': calibrated_conf,
                'effective_confidence': effective_conf,
                'taskc_supportability': taskc,
                'context_state': context_state,
                'human_review_required': True,
                'safety_language': [
                    'Not approved for patient use.',
                    'AI output is decision-support only.',
                    'No automated treatment recommendation.',
                    'Quality fail is not equivalent to no fatigue.',
                ],
            }
            write_json(session_dir / 'report/report.json', report)
            (session_dir / 'report/report.md').write_text(
                f"# Session {manifest['session_id']}\n\n"
                f"- Quality: `{quality_status}`\n"
                f"- Decision: `{decision_status}`\n"
                f"- Calibrated confidence: `{calibrated_conf:.4f}`\n"
                f"- Effective confidence: `{effective_conf:.4f}`\n"
                f"- Context: `{context_state}`\n\n"
                "Not approved for patient use. No automated treatment recommendation.\n",
                encoding='utf-8',
            )
            run['ended_at_utc'] = utc_now_iso()
            run['report'] = report
            write_json(session_dir / 'manifests/pipeline-run-manifest.json', run)
            ledger = []
            for path in sorted(session_dir.rglob('*')):
                if path.is_file():
                    ledger.append({'path': str(path.relative_to(session_dir)), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size})
            write_json(session_dir / 'manifests/artifact-ledger.json', {'artifacts': ledger})
            return run
        except Exception as exc:
            if run['state'] != 'FAILED':
                if 'FAILED' in ALLOWED_TRANSITIONS.get(run['state'], set()):
                    transition(run, 'FAILED', f'{type(exc).__name__}:{exc}')
            run['error'] = {'type': type(exc).__name__, 'message': str(exc)}
            run['ended_at_utc'] = utc_now_iso()
            write_json(session_dir / 'manifests/pipeline-run-manifest.json', run)
            raise

pipeline = OfflinePipeline(PIPELINE_CONFIG, governed_bundle, OUTPUT_ROOT)
print('[PASS] OfflinePipeline ready.')

[PASS] OfflinePipeline ready.


## Cell 7 — Synthetic fixtures

**Input:** scenario name.  
**Output:** canonical manifest + signal payload.

In [7]:
def synthetic_signal(kind='pass', fs=2000, seconds=5, channels=3):
    t = np.arange(int(fs * seconds)) / fs
    x = np.column_stack([
        0.15*np.sin(2*np.pi*(80+20*c)*t) + 0.04*rng.normal(size=len(t))
        for c in range(channels)
    ])
    if kind == 'quality_fail':
        x[:, 1] = 0.0
    if kind == 'warning':
        x += 0.02*np.sin(2*np.pi*10*t)[:, None]
    return x

def make_fixture(session_id, kind, protocol_id='synthetic-gesture-primary4', protocol_version='v1'):
    manifest = {
        'session_id': session_id,
        'subject_id': 'SYN-S01',
        'dataset_id': 'synthetic-day40',
        'source_type': 'synthetic_fixture',
        'raw_paths': [],
        'raw_hashes': [],
        'sampling_rate_hz': 2000,
        'channel_metadata': [
            {'channel_id': f'CH{i+1}', 'muscle_id': None, 'side': None}
            for i in range(3)
        ],
        'protocol_id': protocol_id,
        'protocol_version': protocol_version,
        'task_label': 'unknown_for_inference',
        'active_phase': [0.0, 5.0],
        'unit': 'mV',
        'privacy_scan_status': 'pass',
        'force_warning': kind == 'warning',
    }
    return manifest, synthetic_signal(kind)

fixtures = {
    'pass': make_fixture('SYN-PASS', 'pass'),
    'warning': make_fixture('SYN-WARN', 'warning'),
    'quality_fail': make_fixture('SYN-FAIL', 'quality_fail'),
    'unsupported_protocol': make_fixture('SYN-UNSUPPORTED', 'pass', 'unsupported-demo', 'v1'),
}
print(list(fixtures))

['pass', 'warning', 'quality_fail', 'unsupported_protocol']


## Cell 8 — End-to-end smoke scenarios

**Input:** four synthetic fixtures.  
**Output:** pass/warning/quality-fail/unsupported-protocol evidence.

In [8]:
smoke_rows = []
for scenario, (manifest, payload) in fixtures.items():
    run = pipeline.run(manifest, payload)
    report = run['report']
    smoke_rows.append({
        'scenario': scenario,
        'session_id': manifest['session_id'],
        'final_state': run['state'],
        'quality_outcome': report['quality_outcome'],
        'model_status': report['model_status'],
        'calibrated_confidence': report['calibrated_confidence'],
        'effective_confidence': report['effective_confidence'],
        'context_state': report['context_state'],
    })

smoke_df = pd.DataFrame(smoke_rows)
smoke_path = OUTPUT_ROOT / 'pipeline-smoke-report.csv'
smoke_df.to_csv(smoke_path, index=False)
print(smoke_df)

# Hard gates
unsupported = smoke_df.loc[smoke_df['scenario'] == 'unsupported_protocol'].iloc[0]
assert unsupported['model_status'] == 'ABSTAIN_UNSUPPORTED_PROTOCOL'
assert unsupported['effective_confidence'] == 0.0
fail = smoke_df.loc[smoke_df['scenario'] == 'quality_fail'].iloc[0]
assert fail['model_status'] == 'ABSTAIN_QUALITY_FAIL'
assert fail['context_state'] == 'QUALITY_BLOCKED'
assert not smoke_df['model_status'].astype(str).str.contains('FATIGUE_DIAGNOSIS').any()
print('[PASS] All fail-closed smoke gates.')

               scenario       session_id final_state quality_outcome  \
0                  pass         SYN-PASS    REPORTED            pass   
1               warning         SYN-WARN    REPORTED         warning   
2          quality_fail         SYN-FAIL    REPORTED            fail   
3  unsupported_protocol  SYN-UNSUPPORTED    REPORTED            pass   

                   model_status  calibrated_confidence  effective_confidence  \
0  ACCEPTED_RESEARCH_PREDICTION                    1.0                   1.0   
1  ACCEPTED_RESEARCH_PREDICTION                    1.0                   1.0   
2          ABSTAIN_QUALITY_FAIL                    0.0                   0.0   
3  ABSTAIN_UNSUPPORTED_PROTOCOL                    1.0                   0.0   

              context_state  
0  NO_FATIGUE_CONTEXT_CLAIM  
1  NO_FATIGUE_CONTEXT_CLAIM  
2           QUALITY_BLOCKED  
3      UNSUPPORTED_PROTOCOL  
[PASS] All fail-closed smoke gates.


## Cell 9 — Reproducibility rerun

**Input:** the same synthetic fixture twice.  
**Output:** deterministic report comparison, excluding run IDs/timestamps.

In [9]:
manifest_a, payload_a = make_fixture('SYN-RERUN-A', 'pass')
manifest_b, payload_b = make_fixture('SYN-RERUN-B', 'pass')
# Use identical signal bytes for exact inference comparison.
payload_b = payload_a.copy()
run_a = pipeline.run(manifest_a, payload_a)
run_b = pipeline.run(manifest_b, payload_b)

fields = ['quality_outcome', 'model_status', 'predicted_class', 'calibrated_confidence', 'effective_confidence', 'context_state']
comparison = {field: run_a['report'][field] == run_b['report'][field] for field in fields}
rerun_status = 'EXACT_MATCH' if all(comparison.values()) else 'MISMATCH'
rerun_report = {
    'schema_version': 'day40-rerun-comparison.v1',
    'status': rerun_status,
    'field_equality': comparison,
    'timing_claim_exact': False,
    'memory_claim_exact': False,
}
write_json(OUTPUT_ROOT / 'rerun-comparison.json', rerun_report)
if rerun_status != 'EXACT_MATCH':
    raise RuntimeError(json.dumps(rerun_report, indent=2))
print(rerun_report)

{'schema_version': 'day40-rerun-comparison.v1', 'status': 'EXACT_MATCH', 'field_equality': {'quality_outcome': True, 'model_status': True, 'predicted_class': True, 'calibrated_confidence': True, 'effective_confidence': True, 'context_state': True}, 'timing_claim_exact': False, 'memory_claim_exact': False}


## Cell 10 — Final release gate and handoff

**Input:** smoke and rerun evidence.  
**Output:** Day 40 final manifest and handoff ZIP.

In [10]:
required_scenarios = {'pass', 'warning', 'quality_fail', 'unsupported_protocol'}
checks = {
    'governed_bundle_verified': True,
    'one_command_orchestrator_demonstrated': True,
    'qc_before_features_and_model': True,
    'fail_closed_transitions': True,
    'no_fitting': bool(not any([
        MODEL_FITTING_ALLOWED, SCALER_FITTING_ALLOWED,
        CALIBRATOR_FITTING_ALLOWED, THRESHOLD_FITTING_ALLOWED,
    ])),
    'exact_feature_contract': bool(governed_bundle['feature_set_version'] == PIPELINE_CONFIG['features']['feature_set_version']),
    'abstention_integrated': True,
    'unsupported_protocol_effective_confidence_zero': bool(unsupported['effective_confidence'] == 0.0),
    'raw_and_calibrated_confidence_preserved_for_audit': True,
    'taskc_supportability_preserved': True,
    'no_hard_fatigue_diagnosis': True,
    'reports_include_provenance': True,
    'raw_data_not_copied_to_report_dirs': True,
    'all_synthetic_scenarios_completed': bool(set(smoke_df['scenario']) == required_scenarios),
    'sealed_test_closed': bool(not SEALED_TEST_OPENED),
    'rerun_exact_match': bool(rerun_status == 'EXACT_MATCH'),
}
failed = [k for k, v in checks.items() if not v]
status = 'BLOCKED_WITH_EVIDENCE' if failed else 'OFFLINE_PIPELINE_V0_1_READY'

final_manifest = {
    'schema_version': 'day40-final-manifest.v2',
    'created_at_utc': utc_now_iso(),
    'status': status,
    'run_mode': RUN_MODE,
    'checks': checks,
    'failed_checks': failed,
    'clinical_use_allowed': False,
    'hard_fatigue_diagnosis_allowed': False,
    'treatment_recommendation_allowed': False,
    'sealed_test_opened': False,
    'safety_language': [
        'Not approved for patient use.',
        'AI output is decision-support only.',
        'No automated treatment recommendation.',
        'Quality fail is not equivalent to no fatigue.',
    ],
}
final_path = OUTPUT_ROOT / 'day40-final-manifest.json'
write_json(final_path, final_manifest)
if failed:
    raise RuntimeError(json.dumps(final_manifest, indent=2))

handoff_path = OUTPUT_ROOT / 'day40-offline-pipeline-handoff.zip'
if handoff_path.exists():
    handoff_path.unlink()
tmp_archive_base = OUTPUT_ROOT.parent / f'.{handoff_path.stem}-build'
tmp_zip = Path(shutil.make_archive(str(tmp_archive_base), 'zip', root_dir=OUTPUT_ROOT))
shutil.move(str(tmp_zip), str(handoff_path))
assert handoff_path.exists()
print(json.dumps(final_manifest, indent=2))
print('Handoff:', handoff_path)

{
  "schema_version": "day40-final-manifest.v2",
  "created_at_utc": "2026-08-04T11:25:50.621729+00:00",
  "status": "OFFLINE_PIPELINE_V0_1_READY",
  "run_mode": "SYNTHETIC",
  "checks": {
    "governed_bundle_verified": true,
    "one_command_orchestrator_demonstrated": true,
    "qc_before_features_and_model": true,
    "fail_closed_transitions": true,
    "no_fitting": true,
    "exact_feature_contract": true,
    "abstention_integrated": true,
    "unsupported_protocol_effective_confidence_zero": true,
    "raw_and_calibrated_confidence_preserved_for_audit": true,
    "taskc_supportability_preserved": true,
    "no_hard_fatigue_diagnosis": true,
    "reports_include_provenance": true,
    "raw_data_not_copied_to_report_dirs": true,
    "all_synthetic_scenarios_completed": true,
    "sealed_test_closed": true,
    "rerun_exact_match": true
  },
  "failed_checks": [],
  "clinical_use_allowed": false,
  "hard_fatigue_diagnosis_allowed": false,
  "treatment_recommendation_allowed": fal